In [ ]:
!pip install -U torchao peft transformers huggingface_hub

# 1. Códigos de Chat disponibilizado pela Nvidia

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch

repo_name = "nvidia/Nemotron-Labs-Diffusion-3B"

tokenizer = AutoTokenizer.from_pretrained(repo_name, trust_remote_code=True)
model = AutoModel.from_pretrained(repo_name, trust_remote_code=True)
model = model.cuda().to(torch.bfloat16)

history = []

user_input = input("User: ").strip()
history.append({"role": "user", "content": user_input})

prompt = tokenizer.apply_chat_template(history, tokenize=False, add_generation_prompt=True)
prompt_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device='cuda')

## Chat in AR Mode
# o código disponilizado possue um erro de variavel na linha abaixo,
# mudar de "inputs.input_ids" para "prompt_ids" resolve o problema.
out_ids, nfe = model.ar_generate(inputs.input_ids, max_new_tokens=512)

## Chat in dLM Mode
out_ids, nfe = model.generate(prompt_ids, max_new_tokens=512, block_length=32, threshold=0.9, eos_token_id=tokenizer.eos_token_id)

## Chat in Linear Self-Speculation Mode
out_ids, nfe = model.linear_spec_generate(prompt_ids, max_new_tokens=512, block_length=32, eos_token_id=tokenizer.eos_token_id)

tokenized_out = tokenizer.batch_decode(out_ids[:, prompt_ids.shape[1]:], skip_special_tokens=True)[0]
print(f"Model: {tokenized_out}")
print(f"[Num Function Eval (NFE)={nfe}]")

#### Inferência com "Linear Self-Speculation + LoRA-enhanced Drafter" para aumentar a taxa de aceitação / comprimento de aceitação.

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
from peft import PeftModel

repo = "nvidia/Nemotron-Labs-Diffusion-3B"
tokenizer = AutoTokenizer.from_pretrained(repo, trust_remote_code=True)
model = AutoModel.from_pretrained(repo, trust_remote_code=True)
model = model.cuda().to(torch.bfloat16)

# Attach the linear_spec LoRA adapter.
model = PeftModel.from_pretrained(model, repo, subfolder="linear_spec_lora").eval()
# Unwrap so we can call linear_spec_generate directly (it toggles LoRA internally).
base = model.model

history = [{"role": "user", "content": "Solve: What is 15% of 240?"}]
prompt = tokenizer.apply_chat_template(history, tokenize=False, add_generation_prompt=True)
prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.cuda()

out_ids, nfe = base.linear_spec_generate(
    prompt_ids, max_new_tokens=512, block_length=32,
    eos_token_id=tokenizer.eos_token_id,
)
print(tokenizer.decode(out_ids[0, prompt_ids.shape[1]:], skip_special_tokens=True))
print(f"[NFE={nfe}]")

# 2. Código adaptado para eficiencia no colab.
#### Baixe o modelo na celula abaixo, posteriormente execute o chat na celula seguinte

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download
from transformers import AutoModel
import torch

repo = "nvidia/Nemotron-Labs-Diffusion-3B"
LOCAL_REPO_DIR = Path.cwd() / "hf_repo"

# Baixa o repositorio para a pasta atual (usa o cache se ja existir).
local_repo = snapshot_download(
    repo_id=repo,
    local_dir=str(LOCAL_REPO_DIR),
    local_dir_use_symlinks=False,
    resume_download=True,
 )
print(f"Repo local: {local_repo}")

if not torch.cuda.is_available():
    raise RuntimeError("GPU nao encontrada. Verifique o CUDA no ambiente.")
DEVICE = torch.device("cuda")
MODEL_DTYPE = torch.bfloat16
print(f"Device ativo: {DEVICE}")
print(f"DType do modelo: {MODEL_DTYPE}")

def load_base_model():
    model = AutoModel.from_pretrained(local_repo, trust_remote_code=True)
    return model.to(device=DEVICE, dtype=MODEL_DTYPE)

## Chat com o modelo.
#### Escolha entre um dos modos do modelo para conversar, evite encerrar a celula pelo botão pois a não limpeza da memoria de video pode causar erros de OOM se executadas novamente.

In [ ]:
import torch
import gc
import time
from transformers import AutoTokenizer
from peft import PeftModel

def run_interactive_chat():
    lora_subfolder = "linear_spec_lora"

    print("- Configuracao de Modo de Teste:")
    print("[1] AR (Autoregressive - Padrao)")
    print("[2] dLM (Diffusion Language Model)")
    print("[3] Linear Speculation (Nativo / Sem LoRA)")
    print("[4] Linear Speculation (Otimizado / Com LoRA)")

    mode = input("Escolha o modo para esta sessao (1, 2, 3, 4): ").strip() or "1"
    if mode not in ["1", "2", "3", "4"]:
        mode = "1"

    modes_dict = {
        "1": "Autoregressive",
        "2": "dLM",
        "3": "Linear Speculation (Sem LoRA)",
        "4": "Linear Speculation (Com LoRA)",
    }
    print(f"Modo ativo para testes: {modes_dict[mode]}\n")

    max_tokens = 512

    print("Carregando tokenizer e modelo base...")
    tokenizer = AutoTokenizer.from_pretrained(local_repo, trust_remote_code=True)
    model = load_base_model()

    lora_model = None
    use_lora = False
    if mode == "4":
        print("Carregando adaptador LoRA para o Drafter de Difusao...")
        lora_model = PeftModel.from_pretrained(
            model,
            local_repo,
            subfolder=lora_subfolder,
            is_trainable=False,
        ).eval()
        active_model = lora_model.model
        use_lora = True
    else:
        active_model = model  # modelo base puro para os modos 1, 2 e 3

    history = []
    print("\n--- ChatBot ---")
    print("Digite 'sair' para sair.")

    try:
        while True:
            user_input = input("User: ").strip()
            if user_input.lower() == "sair":
                break
            if not user_input:
                continue

            history.append({"role": "user", "content": user_input})

            prompt = tokenizer.apply_chat_template(history, tokenize=False, add_generation_prompt=True)
            prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)

            start_time = time.time()

            if mode == "1":
                out_ids, nfe = active_model.ar_generate(prompt_ids, max_new_tokens=max_tokens)

            elif mode == "2":
                out_ids, nfe = active_model.generate(
                    prompt_ids,
                    max_new_tokens=max_tokens,
                    block_length=32,
                    threshold=0.9,
                    eos_token_id=tokenizer.eos_token_id,
                )

            elif mode == "3":
                # Modo Especulativo Puro (Nativo do modelo base)
                out_ids, nfe = active_model.linear_spec_generate(
                    prompt_ids,
                    max_new_tokens=max_tokens,
                    block_length=32,
                    eos_token_id=tokenizer.eos_token_id,
                )

            elif mode == "4":
                # Modo Especulativo com o ganho do LoRA Drafter
                out_ids, nfe = active_model.linear_spec_generate(
                    prompt_ids,
                    max_new_tokens=max_tokens,
                    block_length=32,
                    eos_token_id=tokenizer.eos_token_id,
                )

            end_time = time.time()

            # Extracao e decodificacao
            new_tokens_ids = out_ids[0, prompt_ids.shape[1]:]
            response_text = tokenizer.decode(new_tokens_ids, skip_special_tokens=True)

            print(f"Model: {response_text}")

            # Metricas
            duration = end_time - start_time
            num_tokens = len(new_tokens_ids)
            tks = num_tokens / duration if duration > 0 else 0

            print(
                f"[Metricas: {num_tokens} tokens | {duration:.2f}s | "
                f"{tks:.2f} TK/s | NFE={nfe} | LoRA={use_lora}]\n"
)

            history.append({"role": "assistant", "content": response_text})

    except KeyboardInterrupt:
        print("\nInterrompido")
    finally:
        print("\nLimpando VRAM...")
        if "lora_model" in locals() and lora_model is not None:
            del lora_model
        if "active_model" in locals():
            del active_model
        del model, tokenizer

        gc.collect()
        torch.cuda.empty_cache()
        print("Sessao encerrada")

if __name__ == "__main__":
    run_interactive_chat()